In [3]:
import requests
import zipfile
import os
from pathlib import Path

# Configuración manual de ruta si aún no tienes el módulo config
RAW_DIR = Path("data/raw")
YEARS = [2018, 2024]
# Solo los módulos que necesitas para el análisis de impuestos y gasto
MODULES = ["ingresos", "gastoshogar", "concentradohogar"]

BASE_URL = "https://www.inegi.org.mx/contenidos/programas/enigh/nc/{year}/microdatos/enigh{year}_ns_{module}_csv.zip"

def download_and_extract(year, module):
    url = BASE_URL.format(year=year, module=module)
    zip_name = f"{module}_{year}.zip"
    zip_path = RAW_DIR / zip_name
    final_csv_name = f"{module}_{year}.csv"


    # Evitar descargas duplicadas
    if (RAW_DIR / final_csv_name).exists():
        print(f"⏭️  {final_csv_name} ya existe. Saltando...")
        return

    try:
        response = requests.get(url, stream=True)
        if response.status_code != 200:
            print(f"⚠️  {module} {year} no disponible en INEGI.")
            return

        print(f"⬇️  Descargando {module} {year}...")
        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            for file_name in zip_ref.namelist():
                if file_name.lower().endswith(".csv"):
                    zip_ref.extract(file_name, RAW_DIR)
                    # Mover y renombrar a la raíz de /raw
                    (RAW_DIR / file_name).replace(RAW_DIR / final_csv_name)
                    print(f"   ✅ Guardado como: {final_csv_name}")

        zip_path.unlink() # Borrar el zip
        
    except Exception as e:
        print(f"❌ Error en {module} {year}: {e}")

if __name__ == "__main__":
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    print("🚀 Iniciando descarga de microdatos ENIGH...\n")
    for year in YEARS:
        for module in MODULES:
            download_and_extract(year, module)
    print("\n✨ Archivos listos en data/raw.")

🚀 Iniciando descarga de microdatos ENIGH...

⬇️  Descargando ingresos 2018...
   ✅ Guardado como: ingresos_2018.csv
⬇️  Descargando gastoshogar 2018...
   ✅ Guardado como: gastoshogar_2018.csv
⬇️  Descargando concentradohogar 2018...
   ✅ Guardado como: concentradohogar_2018.csv
⬇️  Descargando ingresos 2024...
   ✅ Guardado como: ingresos_2024.csv
⬇️  Descargando gastoshogar 2024...
   ✅ Guardado como: gastoshogar_2024.csv
⬇️  Descargando concentradohogar 2024...
   ✅ Guardado como: concentradohogar_2024.csv

✨ Archivos listos en data/raw.
